In [1]:
# Cell 1 — GPU Check
!nvidia-smi

Mon Apr 27 06:19:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — Install Packages
!pip install -q transformers datasets peft trl bitsandbytes accelerate
!pip install -q huggingface_hub sentencepiece evaluate rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [3]:
# Cell 3 — CUDA Check
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

CUDA available: True
GPU: Tesla T4


In [4]:
# Cell 4 — HuggingFace Login
# Get token: huggingface.co -> Settings -> Access Tokens -> New Token (Write)
from huggingface_hub import login
login(token='hf....................')  # Replace with your token

In [7]:
# Cell 5 — Legal Dataset Create (60 examples)
from datasets import Dataset

qa_pairs = [
    ('What is anticipatory breach of contract?',
     'Anticipatory Breach (Section 39, Indian Contract Act):\nOccurs when a party declares inability or unwillingness to perform before the due date.\n\nTypes:\n1. Express: Party explicitly states they will not perform\n2. Implied: Party actions make performance impossible\n\nRights of aggrieved party:\n1. Treat contract as rescinded immediately\n2. Sue for damages right away\n3. Wait until due date and then sue\n\nExample: A agrees to deliver goods on June 1st.\nOn May 15th, A says I will not deliver.\nB can immediately sue without waiting till June 1st.'),

    ('Explain promissory estoppel in Indian law',
     'Promissory Estoppel:\nPrevents a party from going back on a promise when other party has acted on it.\n\nEssential elements:\n1. Clear and unambiguous promise\n2. Promisee acted in reliance on promise\n3. Detriment suffered if promise not kept\n\nIndian position:\n- Adopted through equity jurisdiction\n- Section 115, Indian Evidence Act covers estoppel\n- Supreme Court applied against government in Indo-Afghan Agencies case\n\nLimitations:\n- Cannot be used as sword, only as shield\n- Does not apply against statute'),

    ('What are the rights of an unpaid seller?',
     'Unpaid Seller Rights (Sale of Goods Act 1930):\n\nRights against GOODS:\n1. Lien (Section 47): Retain goods until payment\n2. Stoppage in Transit (Section 50): Stop goods if buyer insolvent\n3. Right of Resale (Section 54): Resell if buyer defaults\n\nRights against BUYER:\n1. Suit for price (Section 55)\n2. Suit for damages (Section 56)\n3. Suit for interest on delayed payment'),

    ('What is free consent under Indian Contract Act?',
     'Free Consent (Section 14):\nConsent is NOT free when caused by:\n\n1. Coercion (Section 15): Threatening to commit IPC offence\n2. Undue Influence (Section 16): Dominating another will\n3. Fraud (Section 17): False representation knowingly\n4. Misrepresentation (Section 18): Innocent false statement\n5. Mistake (Section 20-22): Bilateral mistake of fact = void\n\nEffect: Contract voidable at option of aggrieved party'),

    ('What are conditions and warranties in sale of goods?',
     'Conditions and Warranties (Sale of Goods Act 1930):\n\nCondition (Section 12(2)):\n- Essential stipulation to main purpose\n- Breach: Can repudiate contract and claim damages\n\nWarranty (Section 12(3)):\n- Collateral stipulation\n- Breach: Damages only, cannot repudiate\n\nImplied Conditions:\n1. Right to sell (Section 14a)\n2. Fitness for purpose (Section 16(1))\n3. Merchantable quality (Section 16(2))\n4. Sale by sample corresponds to bulk (Section 17)'),

    ('Explain consideration under Indian Contract Act',
     'Consideration (Section 2(d)):\nWhen at desire of promisor, promisee does or abstains from doing something.\n\nTypes:\n1. Past Consideration: Act done before promise - valid in India\n2. Present Consideration: Simultaneous with promise\n3. Future Consideration: Promise for promise\n\nRules:\n1. Must move at desire of promisor\n2. Can move from promisee or third party\n3. Must be real and lawful\n4. Need not be adequate\n\nExceptions (Section 25):\n- Natural love and affection\n- Past voluntary services\n- Time-barred debt in writing'),

    ('What is law of agency in India?',
     'Agency (Section 182):\nAgent acts for Principal in dealings with third parties.\n\nCreation:\n1. Express appointment\n2. Implied from conduct\n3. Ratification (Section 196)\n4. Agency by necessity\n5. Agency by estoppel\n\nTermination (Section 201):\n1. Revocation by principal\n2. Renunciation by agent\n3. Death or insanity\n4. Completion of business\n\nAgent duties:\n- Follow instructions (Section 211)\n- Act with skill (Section 212)\n- Render accounts (Section 213)'),

    ('What is doctrine of frustration?',
     'Doctrine of Frustration (Section 56):\nPerformance becomes impossible due to unforeseen events after contract formation.\n\nConditions:\n1. Event after contract formation\n2. Not caused by either party\n3. Performance truly impossible\n\nExamples:\n- Hall burns down before concert\n- War making export impossible\n- Land destroyed by flood\n\nEffect: Contract void automatically.\nBoth parties discharged from obligations.'),

    ('What are minor rights in contract law?',
     'Minor Contracts (Mohori Bibee v Dharmodas Ghose 1903):\n\nFundamental rule: Contract with minor = VOID AB INITIO\n\nConsequences:\n1. Minor not bound, cannot be sued\n2. Cannot ratify on attaining majority\n3. No estoppel against minor\n\nExceptions:\n1. Necessaries (Section 68): Estate liable\n2. Beneficial contracts: Minor can enforce\n3. Minor as agent: Principal bound\n\nAge of majority: 18 years (Indian Majority Act 1875)'),

    ('Explain offer and acceptance rules',
     'Valid Offer (Section 2(a)):\nWillingness to do or abstain from doing something to obtain assent.\n\nRules of valid acceptance:\n1. Absolute and unqualified (Section 7)\n2. Must be communicated\n3. Within reasonable time\n4. Before offer lapses\n5. By prescribed mode\n\nCommunication (Section 4):\n- Against proposer: When put in transmission\n- Against acceptor: When reaches proposer\n\nLapse of offer (Section 6):\n- Revocation before acceptance\n- Death or insanity\n- Lapse of time\n- Counter offer = rejection'),
]

# Augment to 60 examples
augmented_data = []
for instruction, output in qa_pairs:
    augmented_data.append({'instruction': instruction, 'input': '', 'output': output})
    for var in [
        'Please explain: ' + instruction,
        'Under Indian law, ' + instruction.lower(),
        'As a law student: ' + instruction,
        'For my exam: ' + instruction,
        'In simple terms, ' + instruction.lower(),
    ]:
        augmented_data.append({'instruction': var, 'input': '', 'output': output})

def format_prompt(example):
    text = "### Instruction:\n" + example["instruction"] + "\n\n### Response:\n" + example["output"]
    return {"text": text}

dataset = Dataset.from_list(augmented_data)
formatted = dataset.map(format_prompt)
split = formatted.train_test_split(test_size=0.1, seed=42)
print("Train:", len(split["train"]), "| Val:", len(split["test"]))

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Train: 54 | Val: 6


In [8]:
# Cell 6 — Load Model with QLoRA 4-bit
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = 'mistralai/Mistral-7B-v0.1'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit... (5-10 mins)')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True
)
print(f'GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
print('Model loaded!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

GPU memory: 3.84 GB
Model loaded!


In [9]:
# Cell 7 — LoRA Config
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

total = model.num_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}')
print(f'Trainable %:      {100*trainable/total:.4f}%')

Total params:     7,283,675,136
Trainable params: 41,943,040
Trainable %:      0.5758%


In [10]:
# Cell 8 — Training
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./legal-mistral',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    bf16=True,
    max_grad_norm=0.3,
    warmup_steps=10,
    lr_scheduler_type='cosine',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    processing_class=tokenizer,
    args=training_args,
)

print('Training started... (~5 mins)')
trainer.train()
print('Training complete!')

Adding EOS to train dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Training started... (~5 mins)


Step,Training Loss


Training complete!


In [11]:
# Cell 9 — Test Model
def ask_legal(question):
    prompt = '### Instruction:\n' + question + '\n\n### Response:\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('### Response:')[-1].strip()

questions = [
    'What is anticipatory breach of contract?',
    'What are the rights of unpaid seller?',
    'Explain consideration in Indian Contract Act'
]

for q in questions:
    print(f'Q: {q}')
    print(f'A: {ask_legal(q)}')
    print('-' * 50)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Q: What is anticipatory breach of contract?
A: Anti. #  The same time, ## What are not be a new and User  A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A
--------------------------------------------------
Q: What are the rights of unpaid seller?
A: United by  #19. The same time, ## What is a new and User  A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A
-------------------

In [12]:
# Cell 10 — Upload to HuggingFace
repo_name = 'Nithish04/legal-mistral-7b-qlora-v2'

print('Uploading model...')
trainer.model.push_to_hub(repo_name)

print('Uploading tokenizer...')
tokenizer.push_to_hub(repo_name)

print('Done! -> https://huggingface.co/' + repo_name)

Uploading model...


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          |  627kB / 83.9MB            

Uploading tokenizer...


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Done! -> https://huggingface.co/Nithish04/legal-mistral-7b-qlora-v2
